# GPT‑2 → supervised reranker → Pareto-safe hard Top‑10

Эта версия продолжает **успешную supervised-ветку** предыдущего запуска и не обучает новые эпохи.

```text
frozen GPT‑2 Top‑64
        ↓
supervised reranker score
        ↓
Pareto sweep по watch / choice / diversity / popularity / OOD
        ↓
жёсткий Top‑10 с глобальным exposure budget
```

Главные изменения:

1. Все ограничения считаются относительно исходной GPT‑2, а не supervised policy.
2. Нарушения guardrails нормируются на соответствующие допуски:

   ```python
   normalized_violation = violation / tolerance
   ```

3. Оптимизируется реальный дискретный Top‑10: используется последовательный greedy slate с marginal diversity и жёсткими лимитами exposure.
4. Вместо ещё одной эпохи проводится Pareto-поиск коэффициентов:

   \[
   s = s_{sup}+\alpha z(\widehat{watch})+\beta z(\log P_{choice})
       +\gamma\,diversity-\lambda\,popularity-\mu z(OOD).
   \]

5. Конфигурация выбирается только среди прошедших все hard guardrails. Если таких нет, остаётся исходная GPT‑2.
6. Test-ячейка присутствует, но `OPEN_TEST=False`. Она открывается один раз после фиксации validation-конфигурации.

Ноутбук использует сохранённые модели и candidate-cache из `psych_constrained_reranker_v1`, поэтому повторно обучать GPT‑2, choice/completion models и supervised reranker не требуется.

> **Версия для репозитория.** Встроенные `base64/zlib`-блоки удалены. Общий feature-store находится в `src/kion_reward_model.py`, а компактный внешний metadata-файл описан в `artifacts/manifest.json`.


## 1. Установка

Запустите один раз в новой среде Colab. После установки перезапуск обычно не требуется.


In [ ]:
%pip install -q "torch>=2.6,<3" "transformers==4.57.1" "catboost>=1.2.8,<1.3" "duckdb>=1.4,<2" "pandas>=2.2,<4" "numpy>=1.26,<3" "pyarrow>=17" "scikit-learn>=1.5,<2" "tqdm>=4.66,<5" "safetensors>=0.4.5" matplotlib


## 2. Пути и параметры

По умолчанию используется исходная GPT‑2 из `T-lab/weights/best_model`. Чтобы проверить другой генератор кандидатов, измените только `CHECKPOINT_PATH` и `RUN_NAME`.

Папка запуска создаётся в Google Drive. Повторный запуск безопасен: готовые CatBoost-модели, candidate-cache и checkpoints реранжировщика загружаются, а не обучаются заново.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

T_LAB = Path('/content/drive/MyDrive/T-lab') if IN_COLAB else Path.cwd()
DATA_PATH = T_LAB / 'dataset'              # поддерживается и папка, и prepared.zip
CHECKPOINT_PATH = T_LAB / 'weights' / 'best_model'
MODEL_CONFIG_JSON = None
STATE_DICT_KEY = None

RUN_NAME = 'psych_constrained_reranker_v1'
RUN_DIR = T_LAB / RUN_NAME  # read/resume artifacts from GPT2_05
PARETO_NAME = 'psych_supervised_pareto_v2'
PARETO_DIR = T_LAB / PARETO_NAME
WORK_DIR = Path('/content') / RUN_NAME if IN_COLAB else Path.cwd() / 'work' / RUN_NAME
RESUME = True

SEED = 42
AUX_TRAIN_CONTEXTS = 40_000
AUX_VALID_CONTEXTS = 10_000
CHOICE_NEGATIVES = 5
CANDIDATE_POOL = 64
CONTEXT_BATCH_SIZE = 64
CACHE_CONTEXT_BATCH = 256

SUPERVISED_EPOCHS = 3
CONSTRAINED_EPOCHS = 2
RERANK_BATCH_SIZE = 256
SUPERVISED_LR = 1e-3
CONSTRAINED_LR = 3e-4
DUAL_LR = 0.05
MAX_DUAL = 20.0
TEMPERATURE = 0.85
CE_COEF = 0.15
KL_COEF = 0.08
MAX_GRAD_NORM = 1.0

EVAL_USERS = 5_000
EVAL_BATCH_SIZE = 512
BOOTSTRAP_REPEATS = 500

PARETO_RANDOM_CONFIGS = 96
PARETO_STAGE2_BASES = 8
PARETO_SEED = 2026
EXPOSURE_NEW_ITEM_ALLOWANCE = 2

# Жёсткие ограничения при выборе checkpoint на official validation.
NDCG_RELATIVE_FLOOR = 0.99
COVERAGE_RELATIVE_FLOOR = 0.99
GINI_ABSOLUTE_TOL = 5e-4
REPEAT_ABSOLUTE_TOL = 2e-3
OOD_ABSOLUTE_TOL = 2e-3

# Мягкие допуски для batch-wise constrained optimization относительно supervised policy.
CONSTRAINT_TOL = {
    'choice': 0.005,
    'novelty': 0.005,
    'diversity': 0.005,
    'repeat': 0.002,
    'ood': 0.002,
    'exposure_hhi': 1e-5,
}

OPEN_TEST = False
INCLUDE_CACHES_IN_ARCHIVE = False


## 3. Импорты, воспроизводимость и папка запуска

В отличие от предыдущего ноутбука существующая папка не считается ошибкой: это позволяет восстановиться после сброса Colab.


In [ ]:
import os, sys, json, math, time, random, shutil, zipfile, hashlib, platform, gc, copy
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import GPT2Config, GPT2LMHeadModel
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.metrics import roc_auc_score, log_loss, mean_absolute_error, mean_squared_error
from IPython.display import display

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Сначала завершите GPT2_05: {RUN_DIR}')
PARETO_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
if DEVICE.type == 'cuda': torch.set_float32_matmul_precision('high')
print('Устройство:', DEVICE, '| папка:', RUN_DIR)

def dump_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


## 4. Данные и аудит доступного контекста

Используются только предыдущие календарные дни пользователя. Official validation остаётся полностью отделённым от обучения и выбора признаков.


In [ ]:
REQUIRED = [
    'train_interactions.csv', 'validation_interactions.csv',
    'validation_examples.npz', 'test_examples.npz', 'item_mapping.json',
    'resolved_config.json', 'split_boundaries.json', 'input_sha256.json'
]
DATA_DIR = WORK_DIR / 'prepared'
DATA_DIR.mkdir(exist_ok=True)

if DATA_PATH.is_dir():
    source_dir = DATA_PATH
elif DATA_PATH.is_file() and DATA_PATH.suffix.lower() == '.zip':
    with zipfile.ZipFile(DATA_PATH) as zf:
        names = {Path(n).name: n for n in zf.namelist() if not n.endswith('/')}
        for name in REQUIRED:
            if name not in names: raise FileNotFoundError(f'В архиве нет {name}')
            target = DATA_DIR / name
            with zf.open(names[name]) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    source_dir = DATA_DIR
else:
    raise FileNotFoundError(DATA_PATH)

for name in REQUIRED:
    src = source_dir / name
    if not src.is_file(): raise FileNotFoundError(src)
    if src.parent != DATA_DIR:
        shutil.copy2(src, DATA_DIR / name)

mapping = json.loads((DATA_DIR/'item_mapping.json').read_text(encoding='utf-8'))
cfg = json.loads((DATA_DIR/'resolved_config.json').read_text(encoding='utf-8'))
boundaries = json.loads((DATA_DIR/'split_boundaries.json').read_text(encoding='utf-8'))
item_to_index = {str(k): int(v) for k, v in mapping['item_id_to_index'].items()}
index_to_item = np.asarray(mapping['index_to_item_id'], dtype=np.int64)
N_ITEMS = len(index_to_item)
PAD = int(mapping['pad_token_id'])
MAX_CONTEXT = int(cfg['data']['training_sequences']['max_length'])

usecols = ['user_id','item_id','last_watch_dt','watched_pct','source_row_index','item_index']
dtypes = {'user_id':'int64','item_id':'int64','watched_pct':'float32',
          'source_row_index':'int64','item_index':'int32'}
interactions = {}
raw_has_time = False
for split in ['train','validation']:
    p = DATA_DIR/f'{split}_interactions.csv'
    probe = pd.read_csv(p, usecols=['last_watch_dt'], nrows=20_000)
    raw_has_time |= probe.last_watch_dt.astype(str).str.contains(r'[T ]\d{2}:', regex=True).any()
    frame = pd.read_csv(p, usecols=usecols, dtype=dtypes)
    frame['event_time'] = pd.to_datetime(frame.pop('last_watch_dt'), errors='raise')
    frame['split'] = split
    frame = frame.sort_values(['user_id','event_time','source_row_index'], kind='stable').reset_index(drop=True)
    interactions[split] = frame
    print(split, f'{len(frame):,}', frame.event_time.min(), frame.event_time.max())

TIME_OF_DAY_AVAILABLE = bool(raw_has_time and any(
    (f.event_time.dt.hour.ne(0) | f.event_time.dt.minute.ne(0)).any() for f in interactions.values()))
context_audit = {
    'time_of_day_available': TIME_OF_DAY_AVAILABLE,
    'session_duration_available': False,
    'reason': ('sub-day timestamps detected' if TIME_OF_DAY_AVAILABLE else
               'last_watch_dt has calendar dates only; hour and true session duration are not identifiable'),
    'duplicate_user_item_pairs': int(sum(f.duplicated(['user_id','item_id']).sum() for f in interactions.values())),
}
dump_json(PARETO_DIR/'context_signal_audit.json', context_audit)
display(pd.DataFrame([context_audit]))


## 5. Метаданные и исторические признаки без утечки будущего

Используется тот же воспроизводимый feature-store и компактный снимок метаданных KION, что и в предыдущем эксперименте.


In [ ]:
# Общий feature-store и метаданные вынесены из ноутбука в отдельные файлы.
def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'kion_reward_model.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Не найден src/kion_reward_model.py. Запускайте ноутбук из клонированного репозитория.'
    )

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
metadata_candidates = [
    PROJECT_ROOT / 'artifacts' / 'kion_reward_item_features.csv',
    Path('/content/drive/MyDrive/T-lab/reward_model_v1/kion_reward_item_features.csv'),
]
METADATA_PATH = next((path for path in metadata_candidates if path.is_file()), None)
if METADATA_PATH is None:
    raise FileNotFoundError(
        'Не найден kion_reward_item_features.csv. Скопируйте его из reward_model_v1 '
        'в artifacts/ или укажите путь в этой ячейке.'
    )

sys.path.insert(0, str(SRC_DIR))
import kion_reward_model as rm

print('Feature-store module:', SRC_DIR / 'kion_reward_model.py')
print('Catalog metadata:', METADATA_PATH)

meta = pd.read_csv(METADATA_PATH)
LIGHT_GENRES = {'комедии','семейное','мультфильм','мультфильмы','мультсериалы','детские',
                'для детей','развлекательные','юмор','стендап','мюзиклы','музыкальные','караоке'}
SERIOUS_GENRES = {'драмы','документальное','исторические','историческое','военные','биография',
                  'научно-популярные','криминал','детективы'}
meta['item_light'] = meta.main_genre.fillna('no_genre').isin(LIGHT_GENRES).astype('float32')
meta['item_serious'] = meta.main_genre.fillna('no_genre').isin(SERIOUS_GENRES).astype('float32')
meta['item_index'] = meta.item_id.astype(str).map(item_to_index).fillna(-1).astype('int32')
meta_by_index = meta.set_index('item_index', drop=False)
genre_by_index = np.full(N_ITEMS, 'no_genre', dtype=object)
light_by_index = np.zeros(N_ITEMS, dtype=np.float32)
serious_by_index = np.zeros(N_ITEMS, dtype=np.float32)
ok = meta.item_index.between(0, N_ITEMS-1)
genre_by_index[meta.loc[ok,'item_index']] = meta.loc[ok,'main_genre'].fillna('no_genre').astype(str)
light_by_index[meta.loc[ok,'item_index']] = meta.loc[ok,'item_light']
serious_by_index[meta.loc[ok,'item_index']] = meta.loc[ok,'item_serious']

db_path = WORK_DIR/'feature_store.duckdb'
if db_path.exists(): db_path.unlink()
store = rm.FeatureStore(db_path, memory_limit='3GB', threads=4)
# Holdout hygiene: validation experiments never ingest test rows.
store.ingest(DATA_DIR, METADATA_PATH, mapping, splits=('train', 'validation'))
store.build_histories(boundaries['test']['start'])
print('Feature store готов:', db_path)


## 6. Замороженная GPT‑2 — только генератор Top‑64

Параметры GPT‑2 переводятся в `requires_grad=False`. Ни один вариант реранжировщика не меняет candidate generator, поэтому сравнение вариантов честное.


In [ ]:
def load_checkpoint(path):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(path)
    if path.is_dir():
        model, info = GPT2LMHeadModel.from_pretrained(
            str(path), local_files_only=True, attn_implementation='eager', output_loading_info=True)
        problems = {k:v for k,v in info.items() if v}
        if problems: raise ValueError(f'Checkpoint загружен не полностью: {problems}')
    else:
        if MODEL_CONFIG_JSON is not None:
            model_cfg = GPT2Config.from_json_file(str(MODEL_CONFIG_JSON))
        else:
            valid_keys = set(GPT2Config().to_dict())
            kwargs = {k:v for k,v in cfg['model'].items() if k in valid_keys}
            model_cfg = GPT2Config(**kwargs)
        model_cfg._attn_implementation = 'eager'
        model = GPT2LMHeadModel(model_cfg)
        if path.suffix == '.safetensors':
            from safetensors.torch import load_file
            state = load_file(str(path), device='cpu')
        else:
            state = torch.load(path, map_location='cpu', weights_only=True)
        if STATE_DICT_KEY is not None:
            state = state[STATE_DICT_KEY]
        elif isinstance(state, dict):
            for key in ['model_state_dict','state_dict','model']:
                if key in state and isinstance(state[key], dict):
                    state = state[key]; break
        if not isinstance(state, dict) or not all(isinstance(v,torch.Tensor) for v in state.values()):
            raise ValueError('Не найден state_dict')
        for prefix in ['module.','_orig_mod.']:
            if state and all(k.startswith(prefix) for k in state):
                state = {k[len(prefix):]:v for k,v in state.items()}
        if model.config.tie_word_embeddings:
            a,b='transformer.wte.weight','lm_head.weight'
            if a in state and b not in state: state[b]=state[a]
            if b in state and a not in state: state[a]=state[b]
        model.load_state_dict(state, strict=True)
    if model.config.vocab_size != N_ITEMS+1 or model.config.pad_token_id != PAD:
        raise ValueError('Словарь checkpoint не совпадает с prepared.zip')
    if model.config.n_positions < MAX_CONTEXT:
        raise ValueError('n_positions меньше длины истории')
    nearby = (path if path.is_dir() else path.parent)/'item_mapping.json'
    if nearby.exists():
        other = json.loads(nearby.read_text(encoding='utf-8'))
        if other['item_id_to_index'] != mapping['item_id_to_index']:
            raise ValueError('Mapping рядом с весами не совпадает с prepared')
    else:
        print('Предупреждение: mapping рядом с весами отсутствует; проверена только размерность словаря.')
    return model.float().to(DEVICE)

candidate_model = load_checkpoint(CHECKPOINT_PATH).eval()
for p in candidate_model.parameters(): p.requires_grad_(False)


In [ ]:
for p in candidate_model.parameters():
    p.requires_grad_(False)
candidate_model.eval()
print(f'Заморожено параметров GPT‑2: {sum(p.numel() for p in candidate_model.parameters()):,}')


## 7. Психологические прокси и временной split

Прокси включают recent incompletion/completion, паузу между просмотрами, переключение жанров, лёгкий/серьёзный контент, повтор и novelty. Время суток автоматически отключается, если в KION сохранена только дата.


In [ ]:
train_df = interactions['train']
train_counts = np.bincount(train_df.loc[train_df.item_index.ge(0),'item_index'], minlength=N_ITEMS).astype(np.int64)
novelty_by_index = -np.log((train_counts + 1) / (train_counts.sum() + N_ITEMS))
novelty_by_index = (novelty_by_index / novelty_by_index.max()).astype(np.float32)

def state_from_prior(prior, cutoff, history):
    cutoff = pd.Timestamp(cutoff)
    recent = prior.tail(5)
    y = pd.to_numeric(recent.watched_pct, errors='coerce').to_numpy(dtype=float)
    y = y[np.isfinite(y)] / 100.0
    idx = recent.item_index.to_numpy(dtype=int)
    idx = idx[(idx >= 0) & (idx < N_ITEMS)]
    genres = genre_by_index[idx]
    switches = float(np.mean(genres[1:] != genres[:-1])) if len(genres) > 1 else 0.0
    last_day = prior.event_time.max() if len(prior) else pd.NaT
    gap = float((cutoff.normalize() - last_day.normalize()).days) if pd.notna(last_day) else 9999.0
    hour = float(cutoff.hour) if TIME_OF_DAY_AVAILABLE else 0.0
    return {
        'dow_sin': float(np.sin(2*np.pi*cutoff.dayofweek/7)),
        'dow_cos': float(np.cos(2*np.pi*cutoff.dayofweek/7)),
        'hour_sin': float(np.sin(2*np.pi*hour/24)) if TIME_OF_DAY_AVAILABLE else 0.0,
        'hour_cos': float(np.cos(2*np.pi*hour/24)) if TIME_OF_DAY_AVAILABLE else 0.0,
        'time_available': float(TIME_OF_DAY_AVAILABLE),
        'gap_days_log': float(np.log1p(max(0.0, min(gap, 9999.0)))),
        'recent_incomplete': float(np.mean(y < .5)) if len(y) else .5,
        'recent_complete': float(np.mean(y >= .8)) if len(y) else .5,
        'recent_genre_switch': switches,
        'recent_light': float(light_by_index[idx].mean()) if len(idx) else 0.0,
        'last_watch_pct': float(y[-1]) if len(y) else .5,
        'history_len_log': float(np.log1p(len(history))),
    }

STATE_COLUMNS = ['dow_sin','dow_cos','hour_sin','hour_cos','time_available','gap_days_log',
                 'recent_incomplete','recent_complete','recent_genre_switch','recent_light',
                 'last_watch_pct','history_len_log']

def sample_contexts(frame, n, start=None, end=None, seed=42):
    f = frame
    users = f.user_id.to_numpy()
    items = f.item_index.to_numpy()
    dates = f.event_time.to_numpy(dtype='datetime64[ns]')
    known = items >= 0
    previous_same_user = np.r_[False, users[1:] == users[:-1]]
    eligible = known & previous_same_user
    if start is not None: eligible &= dates >= np.datetime64(start)
    if end is not None: eligible &= dates <= np.datetime64(end)
    positions = np.flatnonzero(eligible)
    rng = np.random.default_rng(seed)
    rng.shuffle(positions)
    is_new_user = np.r_[True, users[1:] != users[:-1]]
    group_starts = np.maximum.accumulate(np.where(is_new_user, np.arange(len(f)), 0))
    records=[]
    for pos in tqdm(positions, desc='Контексты', leave=False):
        gs = int(group_starts[pos]); cutoff = pd.Timestamp(dates[pos])
        # Строго предыдущие календарные дни: внутри дня порядок KION не содержит времени.
        block = f.iloc[gs:pos]
        prior = block[block.event_time < cutoff.normalize()]
        hist = prior.loc[prior.item_index.ge(0),'item_index'].to_numpy(dtype=np.int32)[-MAX_CONTEXT:]
        if len(hist) == 0: continue
        state = state_from_prior(prior, cutoff, hist)
        rec = {'context_id':len(records),'user_id':int(users[pos]),
               'target_index':int(items[pos]),'target_item_id':int(f.item_id.iloc[pos]),
               'cutoff':cutoff.normalize(),'history':hist,
               'watched':float(f.watched_pct.iloc[pos]/100.0) if np.isfinite(f.watched_pct.iloc[pos]) else np.nan}
        rec.update(state); records.append(rec)
        if len(records) >= n: break
    if len(records) < n: print(f'Получено {len(records):,} контекстов вместо {n:,}')
    return pd.DataFrame(records)

train_end = pd.Timestamp(boundaries['train']['end'])
valid_start = train_end - pd.Timedelta(days=9)
aux_train = sample_contexts(train_df, AUX_TRAIN_CONTEXTS, end=valid_start-pd.Timedelta(days=1), seed=SEED)
aux_valid = sample_contexts(train_df, AUX_VALID_CONTEXTS, start=valid_start, end=train_end, seed=SEED+1)
assert set(aux_train.context_id).isdisjoint(set())
print('Aux train/valid:', len(aux_train), len(aux_valid), 'граница:', valid_start.date())


## 8. Top‑64 кандидата и hard negatives

Для train target добавляется в конец списка, если GPT‑2 не нашла его в Top‑64. Это используется только для supervised/aux обучения. На official validation target никогда не добавляется искусственно.


In [ ]:
def collate_context_rows(rows):
    batch = list(rows)
    width = max(len(x) for x in batch)
    ids = np.full((len(batch), width), PAD, dtype=np.int64)
    mask = np.zeros((len(batch), width), dtype=np.int64)
    for i,x in enumerate(batch):
        ids[i,:len(x)] = x; mask[i,:len(x)] = 1
    return torch.as_tensor(ids,device=DEVICE), torch.as_tensor(mask,device=DEVICE)

@torch.inference_mode()
def reference_top_candidates(contexts, pool=64):
    all_idx=[]; all_scores=[]; target_scores=[]
    candidate_model.eval()
    for start in tqdm(range(0,len(contexts),CONTEXT_BATCH_SIZE), desc='Frozen GPT-2 Top-64'):
        part=contexts.iloc[start:start+CONTEXT_BATCH_SIZE]
        ids,mask=collate_context_rows(part.history.tolist())
        out=candidate_model(input_ids=ids,attention_mask=mask,use_cache=False).logits
        last=mask.sum(1)-1
        scores=out[torch.arange(len(part),device=DEVICE),last,:N_ITEMS].float()
        top=torch.topk(scores,k=pool,dim=1,largest=True,sorted=True)
        target=torch.as_tensor(part.target_index.to_numpy(),device=DEVICE)
        all_idx.append(top.indices.cpu().numpy().astype(np.int32))
        all_scores.append(top.values.cpu().numpy().astype(np.float32))
        target_scores.append(scores.gather(1,target[:,None]).squeeze(1).cpu().numpy().astype(np.float32))
    return np.vstack(all_idx),np.vstack(all_scores),np.concatenate(target_scores)

def build_choice_pairs(contexts, top_idx, top_score, target_score):
    rows=[]
    for r,(ci,cs,ts) in enumerate(zip(top_idx,top_score,target_score)):
        ctx=contexts.iloc[r]; target=int(ctx.target_index)
        target_pos=np.flatnonzero(ci==target)
        target_rank=int(target_pos[0]+1) if len(target_pos) else CANDIDATE_POOL+1
        selected=[(target,float(ts),target_rank,1)]
        for rank,(item,score) in enumerate(zip(ci,cs),1):
            item=int(item)
            if item==target or any(item==x[0] for x in selected): continue
            selected.append((item,float(score),rank,0))
            if len(selected)==CHOICE_NEGATIVES+1: break
        for item,score,rank,label in selected:
            rows.append({'context_id':int(ctx.context_id),'user_id':int(ctx.user_id),
                         'item_index':item,'item_id':int(index_to_item[item]),
                         'cutoff':ctx.cutoff,'event_day':ctx.cutoff,'y':np.nan,
                         'gpt_ref_logit':score,'gpt_ref_rank':rank,
                         'label':label})
    pairs=pd.DataFrame(rows)
    g=pairs.groupby('context_id').gpt_ref_logit
    pairs['gpt_ref_logit_z']=((pairs.gpt_ref_logit-g.transform('mean'))/
                              g.transform('std').fillna(1).clip(lower=1e-4)).astype('float32')
    pairs['gpt_ref_rank_norm']=(pairs.gpt_ref_rank/(CANDIDATE_POOL+1)).clip(0,1).astype('float32')
    return pairs


raw_top_path = RUN_DIR / 'raw_aux_top64.npz'
if RESUME and raw_top_path.exists():
    with np.load(raw_top_path) as z:
        train_top, train_score, train_target_score = z['train_top'], z['train_score'], z['train_target_score']
        valid_top, valid_score, valid_target_score = z['valid_top'], z['valid_score'], z['valid_target_score']
    print('Top-64 aux cache восстановлен.')
else:
    train_top, train_score, train_target_score = reference_top_candidates(aux_train, CANDIDATE_POOL)
    valid_top, valid_score, valid_target_score = reference_top_candidates(aux_valid, CANDIDATE_POOL)
    np.savez_compressed(raw_top_path,
        train_top=train_top, train_score=train_score, train_target_score=train_target_score,
        valid_top=valid_top, valid_score=valid_score, valid_target_score=valid_target_score)
choice_train_pairs = build_choice_pairs(aux_train, train_top, train_score, train_target_score)
choice_valid_pairs = build_choice_pairs(aux_valid, valid_top, valid_score, valid_target_score)
print(choice_train_pairs.shape, choice_valid_pairs.shape)


## 9. Единый конструктор признаков

Все признаки кандидата вычисляются на дату решения. Candidate-cache далее хранит только компактные числовые признаки, компоненты ограничений и ID жанра.


In [ ]:
PSYCH_COLUMNS = STATE_COLUMNS + ['novelty','repeat','item_light','item_serious',
    'fatigue_x_light','recent_light_x_serious']
CHOICE_EXTRA = ['gpt_ref_logit_z','gpt_ref_rank_norm'] + PSYCH_COLUMNS
CHOICE_FEATURES = rm.FEATURES + CHOICE_EXTRA
COMPLETION_FEATURES = rm.FEATURES + PSYCH_COLUMNS
CAT_FEATURES = rm.CAT_FEATURES

def enrich_pairs(pairs,contexts,include_choice=True):
    work=pairs.copy()
    # Несколько кандидатов имеют одинаковый context_id, поэтому восстанавливаем порядок через отдельный row id.
    work=work.reset_index(drop=True); work['pair_row_id']=np.arange(len(work),dtype=np.int64)
    base_input=work[['pair_row_id','user_id','item_id','event_day','cutoff','y']].rename(columns={'pair_row_id':'row_id'})
    store.con.register('_candidate_pairs',base_input)
    try: base=store.con.execute(store.feature_sql('SELECT * FROM _candidate_pairs')).df().sort_values('row_id')
    finally: store.con.unregister('_candidate_pairs')
    base=base.reset_index(drop=True)
    states=contexts.set_index('context_id')[STATE_COLUMNS]
    state_block=states.loc[work.context_id].reset_index(drop=True)
    for c in STATE_COLUMNS: base[c]=state_block[c].to_numpy(dtype=np.float32)
    idx=work.item_index.to_numpy(dtype=int)
    histories=contexts.set_index('context_id').history
    history_sets={int(cid):set(map(int,h)) for cid,h in histories.items()}
    base['novelty']=novelty_by_index[idx]
    base['repeat']=np.fromiter((int(item in history_sets[int(cid)]) for cid,item in zip(work.context_id,idx)),
                               dtype=np.float32,count=len(work))
    base['item_light']=light_by_index[idx]
    base['item_serious']=serious_by_index[idx]
    base['fatigue_x_light']=base.recent_incomplete*base.item_light
    base['recent_light_x_serious']=base.recent_light*base.item_serious
    if include_choice:
        base['gpt_ref_logit_z']=work.gpt_ref_logit_z.to_numpy(dtype=np.float32)
        base['gpt_ref_rank_norm']=work.gpt_ref_rank_norm.to_numpy(dtype=np.float32)
    for c in CAT_FEATURES: base[c]=base[c].fillna('__unknown__').astype(str)
    numeric=[c for c in (CHOICE_FEATURES if include_choice else COMPLETION_FEATURES) if c not in CAT_FEATURES]
    for c in numeric: base[c]=pd.to_numeric(base[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    base['context_id']=work.context_id.to_numpy()
    base['item_index']=idx
    if 'label' in work: base['label']=work.label.to_numpy()
    return base

choice_train=enrich_pairs(choice_train_pairs,aux_train,True)
choice_valid=enrich_pairs(choice_valid_pairs,aux_valid,True)
display(choice_train[CHOICE_FEATURES+['label']].head())


## 10. Choice model с нуля или восстановление

Choice model различает фактический target и пять сложных альтернатив GPT‑2. Это не причинная CTR-модель: непоказанные фильмы являются hard negatives, а не наблюдавшимися отказами.


In [ ]:
choice_path = RUN_DIR / 'choice_model.cbm'
choice_metrics_path = RUN_DIR / 'choice_metrics.json'

if RESUME and choice_path.exists() and choice_metrics_path.exists():
    choice_model = CatBoostClassifier()
    choice_model.load_model(choice_path)
    choice_metrics = json.loads(choice_metrics_path.read_text(encoding='utf-8'))
    print('Choice model восстановлена.')
else:
    raise FileNotFoundError(f'Нет сохранённой choice model: {choice_path}')

if False:  # обучение намеренно отключено в V06
    choice_model = CatBoostClassifier(
        iterations=700, depth=7, learning_rate=.05, loss_function='Logloss', eval_metric='Logloss',
        random_seed=SEED, l2_leaf_reg=5, random_strength=.5, verbose=100,
        allow_writing_files=False, thread_count=4)
    choice_model.fit(
        Pool(choice_train[CHOICE_FEATURES], choice_train.label, cat_features=CAT_FEATURES),
        eval_set=Pool(choice_valid[CHOICE_FEATURES], choice_valid.label, cat_features=CAT_FEATURES),
        early_stopping_rounds=80, use_best_model=True)
    valid_prob = choice_model.predict_proba(choice_valid[CHOICE_FEATURES])[:, 1]
    valid_raw = choice_model.predict(choice_valid[CHOICE_FEATURES], prediction_type='RawFormulaVal')
    audit = choice_valid[['context_id', 'label']].copy()
    audit['prob'] = valid_prob
    audit['raw'] = valid_raw
    top = audit.sort_values(['context_id', 'raw'], ascending=[True, False]).groupby('context_id').head(1)
    choice_metrics = {
        'logloss': float(log_loss(audit.label, audit.prob)),
        'roc_auc': float(roc_auc_score(audit.label, audit.prob)),
        'group_top1_accuracy': float(top.label.mean()),
        'n_train_pairs': len(choice_train), 'n_valid_pairs': len(choice_valid),
    }
    choice_model.save_model(choice_path)
    dump_json(choice_metrics_path, choice_metrics)
display(pd.DataFrame([choice_metrics]))


## 11. Completion model с нуля или восстановление

Модель оценивает `E[watched_pct | фильм выбран]` только по реальным просмотрам.


In [ ]:
def positive_pairs(contexts):
    mask = np.isfinite(contexts.watched.to_numpy(dtype=float))
    c = contexts.loc[mask].reset_index(drop=True)
    return pd.DataFrame({
        'context_id': c.context_id.astype(int), 'user_id': c.user_id.astype(int),
        'item_index': c.target_index.astype(int), 'item_id': c.target_item_id.astype(int),
        'cutoff': c.cutoff, 'event_day': c.cutoff, 'y': c.watched.astype(float),
    })

completion_train_pairs = positive_pairs(aux_train)
completion_valid_pairs = positive_pairs(aux_valid)
completion_train = enrich_pairs(completion_train_pairs, aux_train, False)
completion_valid = enrich_pairs(completion_valid_pairs, aux_valid, False)
completion_train['y'] = completion_train_pairs.y.to_numpy(dtype=np.float32)
completion_valid['y'] = completion_valid_pairs.y.to_numpy(dtype=np.float32)

completion_path = RUN_DIR / 'completion_model.cbm'
completion_metrics_path = RUN_DIR / 'completion_metrics.json'
if RESUME and completion_path.exists() and completion_metrics_path.exists():
    completion_model = CatBoostRegressor()
    completion_model.load_model(completion_path)
    completion_metrics = json.loads(completion_metrics_path.read_text(encoding='utf-8'))
    print('Completion model восстановлена.')
else:
    raise FileNotFoundError(f'Нет сохранённой completion model: {completion_path}')

if False:  # обучение намеренно отключено в V06
    completion_model = CatBoostRegressor(
        iterations=900, depth=7, learning_rate=.05, loss_function='RMSE', eval_metric='RMSE',
        random_seed=SEED, l2_leaf_reg=7, random_strength=.5, verbose=100,
        allow_writing_files=False, thread_count=4)
    completion_model.fit(
        Pool(completion_train[COMPLETION_FEATURES], completion_train.y, cat_features=CAT_FEATURES),
        eval_set=Pool(completion_valid[COMPLETION_FEATURES], completion_valid.y, cat_features=CAT_FEATURES),
        early_stopping_rounds=100, use_best_model=True)
    cp = np.clip(completion_model.predict(completion_valid[COMPLETION_FEATURES]), 0, 1)
    completion_metrics = {
        'MAE': float(mean_absolute_error(completion_valid.y, cp)),
        'RMSE': float(np.sqrt(mean_squared_error(completion_valid.y, cp))),
        'train_mean_MAE': float(mean_absolute_error(
            completion_valid.y, np.full(len(cp), completion_train.y.mean()))),
        'n_train': len(completion_train), 'n_valid': len(completion_valid),
    }
    completion_model.save_model(completion_path)
    dump_json(completion_metrics_path, completion_metrics)
display(pd.DataFrame([completion_metrics]))


## 12. Прозрачный OOD-риск

Штраф строится по выходу числовых признаков за центральный train-диапазон и отсутствующим метаданным. Он заморожен до обучения реранжировщика.


In [ ]:
OOD_COLUMNS = [c for c in COMPLETION_FEATURES if c not in CAT_FEATURES]
ood_path = RUN_DIR / 'ood_support.json'
if RESUME and ood_path.exists():
    ood_stats = json.loads(ood_path.read_text(encoding='utf-8'))
else:
    ood_stats = {}
    for c in OOD_COLUMNS:
        x = pd.to_numeric(choice_train[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        q = x.quantile([.01, .25, .5, .75, .99])
        ood_stats[c] = {f'q{int(k*100):02d}': float(v) for k, v in q.items()}
    dump_json(ood_path, ood_stats)

def ood_risk(frame):
    risks = []
    for c, s in ood_stats.items():
        x = pd.to_numeric(frame[c], errors='coerce').to_numpy(dtype=float)
        scale = max(s['q75'] - s['q25'], 1e-4)
        z = np.maximum(np.maximum((s['q01'] - x) / scale, (x - s['q99']) / scale), 0)
        z = np.where(np.isfinite(z), z, 2.0)
        risks.append(np.clip(z, 0, 5))
    r = np.mean(np.vstack(risks), axis=0)
    missing = np.maximum(frame['i_missing'].to_numpy(float), frame['metadata_missing'].to_numpy(float))
    return np.clip(1 - np.exp(-r) + .25 * missing, 0, 1).astype(np.float32)

display(pd.Series(ood_risk(choice_valid)).describe())


## 13. Official validation и candidate-cache

Тяжёлые признаки и прогнозы замороженных моделей рассчитываются один раз. После сброса Colab достаточно снова выполнить определения выше: существующие cache-файлы будут загружены.


In [ ]:
def contexts_from_npz(path, history_frame, n, seed, cutoff):
    z=np.load(path)
    rng=np.random.default_rng(seed); take=np.arange(len(z['users'])); rng.shuffle(take); take=take[:n]
    hf=history_frame.sort_values(['user_id','event_time','source_row_index'],kind='stable')
    selected_users=set(map(int,z['users'][take]))
    # Материализуем истории только выбранных для аудита пользователей, а не все 130k+ групп.
    selected_history=hf[hf.user_id.isin(selected_users)]
    groups={int(u):g for u,g in selected_history.groupby('user_id',sort=False)}
    records=[]
    for new_id,i in enumerate(tqdm(take,desc='Official contexts',leave=False)):
        user=int(z['users'][i]); length=int(z['lengths'][i])
        hist=z['histories'][i,:length].astype(np.int32)
        prior=groups.get(user,hf.iloc[:0])
        state=state_from_prior(prior,pd.Timestamp(cutoff),hist)
        rec={'context_id':new_id,'user_id':user,'target_index':int(z['targets'][i]),
             'target_item_id':int(index_to_item[int(z['targets'][i])]),'cutoff':pd.Timestamp(cutoff),
             'history':hist,'watched':np.nan}; rec.update(state); records.append(rec)
    return pd.DataFrame(records)

official_valid=contexts_from_npz(DATA_DIR/'validation_examples.npz',train_df,EVAL_USERS,SEED,
                                 boundaries['validation']['start'])


In [ ]:
official_valid = contexts_from_npz(
    DATA_DIR / 'validation_examples.npz', train_df, EVAL_USERS, SEED,
    boundaries['validation']['start'])

RERANK_FEATURES = [
    'gpt_logit_z', 'gpt_rank_norm', 'log_choice', 'watched_pct', 'novelty',
    'repeat', 'ood', 'item_light', 'item_serious', 'fatigue_x_light',
    'recent_light_x_serious',
] + STATE_COLUMNS
COMPONENT_NAMES = ['log_choice', 'watched_pct', 'novelty', 'repeat', 'ood']

genre_values = sorted(set(map(str, genre_by_index.tolist())))
genre_to_code = {g: i for i, g in enumerate(genre_values)}
genre_code_by_index = np.asarray([genre_to_code[str(g)] for g in genre_by_index], dtype=np.int16)

def prepare_candidates(contexts, top_idx, top_scores, target_scores, inject_target):
    items = top_idx.copy().astype(np.int32)
    scores = top_scores.copy().astype(np.float32)
    ranks = np.tile(np.arange(1, items.shape[1] + 1, dtype=np.int16), (len(items), 1))
    target_pos = np.full(len(items), -1, dtype=np.int16)
    natural_hit = np.zeros(len(items), dtype=np.int8)
    for i, target in enumerate(contexts.target_index.to_numpy(dtype=np.int32)):
        found = np.flatnonzero(items[i] == target)
        if len(found):
            target_pos[i] = int(found[0]); natural_hit[i] = 1
        elif inject_target:
            items[i, -1] = target
            scores[i, -1] = target_scores[i]
            ranks[i, -1] = CANDIDATE_POOL + 1
            target_pos[i] = CANDIDATE_POOL - 1
    return items, scores, ranks, target_pos, natural_hit

def candidate_pairs(contexts, items, scores, ranks):
    b, k = items.shape
    mean = scores.mean(1, keepdims=True)
    std = np.maximum(scores.std(1, keepdims=True), 1e-4)
    repeated = contexts.loc[contexts.index.repeat(k)].reset_index(drop=True)
    flat_items = items.reshape(-1)
    return pd.DataFrame({
        'context_id': repeated.context_id.astype(int),
        'user_id': repeated.user_id.astype(int),
        'item_index': flat_items,
        'item_id': index_to_item[flat_items],
        'cutoff': repeated.cutoff,
        'event_day': repeated.cutoff,
        'y': np.nan,
        'gpt_ref_logit_z': ((scores - mean) / std).reshape(-1).astype(np.float32),
        'gpt_ref_rank_norm': (ranks / (CANDIDATE_POOL + 1)).reshape(-1).astype(np.float32),
    })

def build_candidate_cache(contexts, top_idx, top_scores, target_scores, path, inject_target):
    if RESUME and path.exists():
        print('Загружаем cache:', path.name)
        with np.load(path) as z:
            return {name: z[name] for name in z.files}

    items, scores, ranks, target_pos, natural_hit = prepare_candidates(
        contexts, top_idx, top_scores, target_scores, inject_target)
    all_features, all_components, all_genres = [], [], []

    for start in tqdm(range(0, len(contexts), CACHE_CONTEXT_BATCH), desc=f'Features {path.stem}'):
        end = min(start + CACHE_CONTEXT_BATCH, len(contexts))
        part = contexts.iloc[start:end].reset_index(drop=True).copy()
        # Внутри chunk context_id должен быть локально однозначным.
        part['context_id'] = np.arange(len(part), dtype=np.int64)
        pairs = candidate_pairs(part, items[start:end], scores[start:end], ranks[start:end])
        frame = enrich_pairs(pairs, part, True)
        raw_choice = choice_model.predict(frame[CHOICE_FEATURES], prediction_type='RawFormulaVal')
        log_choice = np.log(np.clip(1 / (1 + np.exp(-np.clip(raw_choice, -30, 30))), 1e-6, 1))
        watched = np.clip(completion_model.predict(frame[COMPLETION_FEATURES]), 0, 1)
        ood = ood_risk(frame)

        block = {
            'gpt_logit_z': frame.gpt_ref_logit_z.to_numpy(np.float32),
            'gpt_rank_norm': frame.gpt_ref_rank_norm.to_numpy(np.float32),
            'log_choice': log_choice.astype(np.float32),
            'watched_pct': watched.astype(np.float32),
            'novelty': frame.novelty.to_numpy(np.float32),
            'repeat': frame.repeat.to_numpy(np.float32),
            'ood': ood,
            'item_light': frame.item_light.to_numpy(np.float32),
            'item_serious': frame.item_serious.to_numpy(np.float32),
            'fatigue_x_light': frame.fatigue_x_light.to_numpy(np.float32),
            'recent_light_x_serious': frame.recent_light_x_serious.to_numpy(np.float32),
        }
        for c in STATE_COLUMNS:
            block[c] = frame[c].to_numpy(np.float32)
        feature_matrix = np.column_stack([block[c] for c in RERANK_FEATURES])
        component_matrix = np.column_stack([block[c] for c in COMPONENT_NAMES])
        b = end - start
        all_features.append(feature_matrix.reshape(b, CANDIDATE_POOL, -1).astype(np.float16))
        all_components.append(component_matrix.reshape(b, CANDIDATE_POOL, -1).astype(np.float16))
        all_genres.append(genre_code_by_index[items[start:end]])

    payload = {
        'items': items, 'features': np.concatenate(all_features),
        'components': np.concatenate(all_components), 'genres': np.concatenate(all_genres),
        'target_pos': target_pos, 'targets': contexts.target_index.to_numpy(np.int32),
        'users': contexts.user_id.to_numpy(np.int64),
        'natural_hit64': natural_hit,
    }
    np.savez_compressed(path, **payload)
    print('Сохранён cache:', path, f'{path.stat().st_size/2**20:.1f} MB')
    return payload

valid_top_off, valid_score_off, valid_target_score_off = reference_top_candidates(
    official_valid, CANDIDATE_POOL)

train_cache = build_candidate_cache(
    aux_train, train_top, train_score, train_target_score,
    RUN_DIR / 'candidate_train.npz', inject_target=True)
aux_valid_cache = build_candidate_cache(
    aux_valid, valid_top, valid_score, valid_target_score,
    RUN_DIR / 'candidate_aux_valid.npz', inject_target=True)
official_cache = build_candidate_cache(
    official_valid, valid_top_off, valid_score_off, valid_target_score_off,
    RUN_DIR / 'candidate_official_valid.npz', inject_target=False)

print('Natural candidate Recall@64:', float(official_cache['natural_hit64'].mean()))


## 14. Neural reranker и supervised warm start

Warm start обучается listwise cross-entropy: target должен получить наибольший score среди 64 кандидатов. Genre embedding и компактные числовые признаки обучаются с нуля.


In [ ]:
norm_path = RUN_DIR / 'reranker_normalization.json'
if RESUME and norm_path.exists():
    norm = json.loads(norm_path.read_text(encoding='utf-8'))
    feature_mean = np.asarray(norm['mean'], dtype=np.float32)
    feature_std = np.asarray(norm['std'], dtype=np.float32)
else:
    raise FileNotFoundError(f'Нет нормировки supervised reranker: {norm_path}')

if False:  # пересчёт намеренно отключён в V06
    flat = train_cache['features'].reshape(-1, len(RERANK_FEATURES))
    rng = np.random.default_rng(SEED)
    take = rng.choice(len(flat), size=min(500_000, len(flat)), replace=False)
    sample = flat[take].astype(np.float32)
    feature_mean = sample.mean(0)
    feature_std = sample.std(0).clip(1e-4)
    dump_json(norm_path, {'features': RERANK_FEATURES,
                          'mean': feature_mean.tolist(), 'std': feature_std.tolist()})

class CandidateDataset(Dataset):
    def __init__(self, cache):
        self.cache = cache
    def __len__(self):
        return len(self.cache['items'])
    def __getitem__(self, i):
        return (
            self.cache['features'][i].astype(np.float32),
            self.cache['genres'][i].astype(np.int64),
            self.cache['items'][i].astype(np.int64),
            self.cache['components'][i].astype(np.float32),
            np.int64(self.cache['target_pos'][i]),
        )

class ConstrainedReranker(nn.Module):
    def __init__(self, n_features, n_genres, genre_dim=8):
        super().__init__()
        self.genre = nn.Embedding(n_genres, genre_dim)
        self.net = nn.Sequential(
            nn.Linear(n_features + genre_dim, 64), nn.GELU(), nn.Dropout(.10),
            nn.Linear(64, 32), nn.GELU(), nn.Dropout(.05), nn.Linear(32, 1))
        self.register_buffer('feature_mean', torch.as_tensor(feature_mean))
        self.register_buffer('feature_std', torch.as_tensor(feature_std))
    def forward(self, features, genres):
        x = (features - self.feature_mean) / self.feature_std
        x = torch.cat([x, self.genre(genres)], dim=-1)
        return self.net(x).squeeze(-1)

def new_reranker():
    return ConstrainedReranker(len(RERANK_FEATURES), len(genre_values)).to(DEVICE)

def move_batch(batch):
    x, genre, items, components, target = batch
    return (x.to(DEVICE), genre.to(DEVICE), items.to(DEVICE),
            components.to(DEVICE), target.to(DEVICE))

train_loader = DataLoader(CandidateDataset(train_cache), batch_size=RERANK_BATCH_SIZE,
                          shuffle=True, num_workers=0, pin_memory=AMP_ENABLED)
aux_loader = DataLoader(CandidateDataset(aux_valid_cache), batch_size=RERANK_BATCH_SIZE,
                        shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)

supervised_path = RUN_DIR / 'reranker_supervised.pt'
supervised_model = new_reranker()
supervised_history = []

if RESUME and supervised_path.exists():
    supervised_model.load_state_dict(torch.load(supervised_path, map_location=DEVICE, weights_only=True))
    print('Supervised reranker восстановлен.')
else:
    raise FileNotFoundError(f'Нет supervised checkpoint: {supervised_path}')

if False:  # дополнительные эпохи намеренно отключены в V06
    optimizer = torch.optim.AdamW(supervised_model.parameters(), lr=SUPERVISED_LR, weight_decay=1e-3)
    best_valid = float('inf')
    for epoch in range(1, SUPERVISED_EPOCHS + 1):
        supervised_model.train(); train_losses = []
        for batch in tqdm(train_loader, desc=f'Supervised {epoch}/{SUPERVISED_EPOCHS}', leave=True):
            x, genre, items, components, target = move_batch(batch)
            optimizer.zero_grad(set_to_none=True)
            logits = supervised_model(x, genre)
            loss = F.cross_entropy(logits, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(supervised_model.parameters(), MAX_GRAD_NORM)
            optimizer.step(); train_losses.append(float(loss.detach()))
        supervised_model.eval(); valid_losses = []
        with torch.inference_mode():
            for batch in aux_loader:
                x, genre, items, components, target = move_batch(batch)
                valid_losses.append(float(F.cross_entropy(supervised_model(x, genre), target)))
        row = {'epoch': epoch, 'train_ce': float(np.mean(train_losses)),
               'valid_ce': float(np.mean(valid_losses))}
        supervised_history.append(row); display(pd.DataFrame([row]))
        if row['valid_ce'] < best_valid:
            best_valid = row['valid_ce']
            torch.save(supervised_model.state_dict(), supervised_path)
    supervised_model.load_state_dict(torch.load(supervised_path, map_location=DEVICE, weights_only=True))

supervised_model.eval()
if supervised_history:
    pd.DataFrame(supervised_history).to_csv(RUN_DIR / 'supervised_training_trace.csv', index=False)


## 15. Метрики Top‑10 и жёсткие guardrails

`CandidateRecall@64` — потолок качества данного candidate generator. Choice-weighted компоненты считаются внутри итогового Top‑10. Exposure HHI используется как дифференцируемый proxy концентрации, а настоящий Gini — как validation guardrail.


In [ ]:
def gini_exposure(items):
    counts = np.bincount(items.ravel(), minlength=N_ITEMS).astype(float)
    if counts.sum() == 0: return 0.0
    x = np.sort(counts); n = len(x)
    return float((2 * np.dot(np.arange(1, n + 1), x) / (n * x.sum())) - (n + 1) / n)

@torch.inference_mode()
def model_order(model, cache):
    model.eval(); orders = []
    loader = DataLoader(CandidateDataset(cache), batch_size=EVAL_BATCH_SIZE,
                        shuffle=False, num_workers=0)
    for batch in loader:
        x, genre, items, components, target = move_batch(batch)
        orders.append(torch.argsort(model(x, genre), dim=1, descending=True).cpu().numpy())
    return np.vstack(orders)

def policy_metrics(cache, order, label, return_per_user=False):
    items = np.take_along_axis(cache['items'], order, axis=1)
    components = np.take_along_axis(cache['components'], order[:, :, None], axis=1).astype(np.float32)
    genres = np.take_along_axis(cache['genres'], order, axis=1)
    targets = cache['targets']
    metrics = {'label': label, 'n_users': len(items),
               'CandidateRecall@64': float(cache['natural_hit64'].mean())}
    per_user = {}
    for k in [10, 20]:
        hit = items[:, :k] == targets[:, None]
        recall = hit.any(1).astype(np.float32)
        rank = np.where(hit, np.arange(1, k + 1), k + 1).min(1)
        ndcg = np.where(rank <= k, 1 / np.log2(rank + 1), 0).astype(np.float32)
        metrics[f'Recall@{k}'] = float(recall.mean())
        metrics[f'NDCG@{k}'] = float(ndcg.mean())
        per_user[f'Recall@{k}'] = recall
        per_user[f'NDCG@{k}'] = ndcg

    top_items = items[:, :10]
    top_comp = components[:, :10]
    top_genres = genres[:, :10]
    choice_logits = top_comp[:, :, COMPONENT_NAMES.index('log_choice')]
    choice_weights = np.exp(choice_logits - choice_logits.max(1, keepdims=True))
    choice_weights /= choice_weights.sum(1, keepdims=True)

    for name in COMPONENT_NAMES:
        values = top_comp[:, :, COMPONENT_NAMES.index(name)]
        user_values = (choice_weights * values).sum(1)
        metrics[f'component_{name}'] = float(user_values.mean())
        per_user[f'component_{name}'] = user_values

    diversity = (top_genres[:, :, None] != top_genres[:, None, :]).mean((1, 2)).astype(np.float32)
    metrics['component_diversity'] = float(diversity.mean())
    per_user['component_diversity'] = diversity
    metrics['Coverage@10'] = float(len(np.unique(top_items)) / N_ITEMS)
    metrics['ExposureGini@10'] = gini_exposure(top_items)
    counts = np.bincount(top_items.ravel(), minlength=N_ITEMS).astype(float)
    exposure = counts / max(counts.sum(), 1)
    metrics['ExposureHHI@10'] = float(np.square(exposure).sum())
    return (metrics, per_user) if return_per_user else metrics

gpt_order = np.tile(np.arange(CANDIDATE_POOL), (len(official_cache['items']), 1))
baseline_metrics, baseline_per_user = policy_metrics(
    official_cache, gpt_order, '01_GPT2', return_per_user=True)
supervised_order = model_order(supervised_model, official_cache)
supervised_metrics, supervised_per_user = policy_metrics(
    official_cache, supervised_order, '02_supervised', return_per_user=True)

def hard_guard(metrics, baseline=baseline_metrics):
    checks = {
        'ndcg': metrics['NDCG@10'] >= baseline['NDCG@10'] * NDCG_RELATIVE_FLOOR,
        'coverage': metrics['Coverage@10'] >= baseline['Coverage@10'] * COVERAGE_RELATIVE_FLOOR,
        'gini': metrics['ExposureGini@10'] <= baseline['ExposureGini@10'] + GINI_ABSOLUTE_TOL,
        'choice': metrics['component_log_choice'] >= baseline['component_log_choice'] - CONSTRAINT_TOL['choice'],
        'novelty': metrics['component_novelty'] >= baseline['component_novelty'] - CONSTRAINT_TOL['novelty'],
        'diversity': metrics['component_diversity'] >= baseline['component_diversity'] - CONSTRAINT_TOL['diversity'],
        'repeat': metrics['component_repeat'] <= baseline['component_repeat'] + REPEAT_ABSOLUTE_TOL,
        'ood': metrics['component_ood'] <= baseline['component_ood'] + OOD_ABSOLUTE_TOL,
    }
    return bool(all(checks.values())), checks

display(pd.DataFrame([baseline_metrics, supervised_metrics]))


## 16. Замороженные scores и признаки Pareto-поиска

Все компоненты уже находятся в `candidate_official_valid.npz`. Популярность определяется через реальную концентрацию исходного GPT‑2 Top‑10: фильмы, получившие больше baseline-exposure, получают больший penalty.


In [ ]:
@torch.inference_mode()
def model_scores(model, cache):
    model.eval(); outputs = []
    loader = DataLoader(CandidateDataset(cache), batch_size=EVAL_BATCH_SIZE,
                        shuffle=False, num_workers=0)
    for batch in loader:
        x, genre, items, components, target = move_batch(batch)
        outputs.append(model(x, genre).cpu().numpy().astype(np.float32))
    return np.vstack(outputs)

def slate_z(values):
    values = np.asarray(values, dtype=np.float32)
    mean = values.mean(1, keepdims=True)
    std = values.std(1, keepdims=True).clip(1e-4)
    return (values - mean) / std

items_valid = official_cache['items'].astype(np.int32)
genres_valid = official_cache['genres'].astype(np.int16)
components_valid = official_cache['components'].astype(np.float32)
supervised_scores_valid = model_scores(supervised_model, official_cache)

component_index = {name: i for i, name in enumerate(COMPONENT_NAMES)}
watch_valid = components_valid[:, :, component_index['watched_pct']]
choice_valid_matrix = components_valid[:, :, component_index['log_choice']]
novelty_valid = components_valid[:, :, component_index['novelty']]
repeat_valid = components_valid[:, :, component_index['repeat']]
ood_valid = components_valid[:, :, component_index['ood']]

gpt_top10 = items_valid[:, :10]
gpt_exposure_counts = np.bincount(gpt_top10.ravel(), minlength=N_ITEMS).astype(np.int64)
gpt_exposure_pressure = np.log1p(gpt_exposure_counts).astype(np.float32)
gpt_exposure_pressure /= max(float(gpt_exposure_pressure.max()), 1.0)

Z_SUP = slate_z(supervised_scores_valid)
Z_WATCH = slate_z(watch_valid)
Z_CHOICE = slate_z(choice_valid_matrix)
Z_OOD = slate_z(ood_valid)
POPULARITY_PRESSURE = gpt_exposure_pressure[items_valid]

print('Validation users:', len(items_valid))
print('Candidate Recall@64:', baseline_metrics['CandidateRecall@64'])


## 17. Нормированные нарушения относительно GPT‑2

`guard_violation = 0` означает, что политика прошла все ограничения. Значение `1` означает нарушение одного guardrail ровно на величину его допуска. Благодаря нормировке NDCG, Gini и OOD участвуют в сравнении в сопоставимом масштабе.


In [ ]:
def normalized_guard_violations(metrics, baseline=baseline_metrics):
    tolerances = {
        'ndcg': max(baseline['NDCG@10'] * (1 - NDCG_RELATIVE_FLOOR), 1e-8),
        'coverage': max(baseline['Coverage@10'] * (1 - COVERAGE_RELATIVE_FLOOR), 1e-8),
        'gini': GINI_ABSOLUTE_TOL,
        'hhi': CONSTRAINT_TOL['exposure_hhi'],
        'choice': CONSTRAINT_TOL['choice'],
        'novelty': CONSTRAINT_TOL['novelty'],
        'diversity': CONSTRAINT_TOL['diversity'],
        'repeat': REPEAT_ABSOLUTE_TOL,
        'ood': OOD_ABSOLUTE_TOL,
    }
    raw = {
        'ndcg': baseline['NDCG@10'] * NDCG_RELATIVE_FLOOR - metrics['NDCG@10'],
        'coverage': baseline['Coverage@10'] * COVERAGE_RELATIVE_FLOOR - metrics['Coverage@10'],
        'gini': metrics['ExposureGini@10'] - baseline['ExposureGini@10'] - GINI_ABSOLUTE_TOL,
        'hhi': metrics['ExposureHHI@10'] - baseline['ExposureHHI@10'] - CONSTRAINT_TOL['exposure_hhi'],
        'choice': baseline['component_log_choice'] - CONSTRAINT_TOL['choice'] - metrics['component_log_choice'],
        'novelty': baseline['component_novelty'] - CONSTRAINT_TOL['novelty'] - metrics['component_novelty'],
        'diversity': baseline['component_diversity'] - CONSTRAINT_TOL['diversity'] - metrics['component_diversity'],
        'repeat': metrics['component_repeat'] - baseline['component_repeat'] - REPEAT_ABSOLUTE_TOL,
        'ood': metrics['component_ood'] - baseline['component_ood'] - OOD_ABSOLUTE_TOL,
    }
    normalized = {name: max(0.0, raw[name] / tolerances[name]) for name in raw}
    return normalized, float(sum(normalized.values()))

base_violations, base_violation_sum = normalized_guard_violations(baseline_metrics)
assert base_violation_sum == 0.0
sup_violations, sup_violation_sum = normalized_guard_violations(supervised_metrics)
display(pd.DataFrame([{'policy': 'GPT2', **base_violations, 'sum': base_violation_sum},
                      {'policy': 'supervised', **sup_violations, 'sum': sup_violation_sum}]))


## 18. Stage 1 — быстрый Pareto-поиск статических коэффициентов

Сначала проверяются десятки сочетаний `α, β, λ, μ` без дорогого последовательного slate-прохода. В следующий этап попадают только лучшие feasible/Pareto-кандидаты и варианты с минимальным нормированным нарушением.


In [ ]:
def static_score(config):
    return (Z_SUP
            + config['alpha_watch'] * Z_WATCH
            + config['beta_choice'] * Z_CHOICE
            - config['lambda_popularity'] * POPULARITY_PRESSURE
            - config['mu_ood'] * Z_OOD)

def order_from_score(scores):
    return np.argsort(-scores, axis=1, kind='stable')

rng = np.random.default_rng(PARETO_SEED)
stage1_configs = [{
    'config_id': 'supervised_zero', 'alpha_watch': 0.0, 'beta_choice': 0.0,
    'lambda_popularity': 0.0, 'mu_ood': 0.0,
}]
for i in range(PARETO_RANDOM_CONFIGS):
    stage1_configs.append({
        'config_id': f'random_{i:03d}',
        'alpha_watch': float(rng.uniform(0.0, 0.75)),
        'beta_choice': float(rng.uniform(0.0, 0.40)),
        'lambda_popularity': float(rng.uniform(0.0, 0.50)),
        'mu_ood': float(rng.uniform(0.0, 0.30)),
    })

stage1_rows = []
for cfg in tqdm(stage1_configs, desc='Pareto stage 1'):
    metrics = policy_metrics(official_cache, order_from_score(static_score(cfg)), cfg['config_id'])
    violations, violation_sum = normalized_guard_violations(metrics)
    stage1_rows.append({**cfg, **metrics, 'guard_violation': violation_sum,
                        'hard_feasible': violation_sum == 0.0,
                        **{f'v_{k}': v for k, v in violations.items()}})

stage1 = pd.DataFrame(stage1_rows)
stage1.to_csv(PARETO_DIR / 'pareto_stage1.csv', index=False)
display(stage1.sort_values(['guard_violation', 'component_watched_pct'], ascending=[True, False]).head(15))


## 19. Stage 2 — жёсткий Top‑10, marginal diversity и глобальный exposure budget

Пользователи обрабатываются в фиксированном случайном порядке. Для каждого фильма задаётся бюджет относительно его exposure в исходной GPT‑2. Уже исчерпавший бюджет фильм исключается, пока существуют допустимые альтернативы. После выбора Top‑10 остальные кандидаты добавляются в исходном score-порядке только для расчёта Recall/NDCG@20.


In [ ]:
def pick_stage1_bases(frame, n=PARETO_STAGE2_BASES):
    feasible = frame[frame.hard_feasible].sort_values('component_watched_pct', ascending=False)
    low_violation = frame.sort_values(['guard_violation', 'component_watched_pct'], ascending=[True, False])
    picked = pd.concat([feasible.head(n), low_violation.head(n)]).drop_duplicates('config_id').head(n)
    return picked

def hard_top10_order(scores, items, genres, repeat_matrix, gamma_diversity,
                     budget_relax, user_order_seed):
    n, k = items.shape
    total_slots = n * 10
    scale = total_slots / max(int(gpt_exposure_counts.sum()), 1)
    budget = np.ceil(gpt_exposure_counts * scale * (1 + budget_relax)
                     + EXPOSURE_NEW_ITEM_ALLOWANCE).astype(np.int64)
    counts = np.zeros(N_ITEMS, dtype=np.int64)
    output = np.empty((n, k), dtype=np.int16)
    user_order = np.random.default_rng(user_order_seed).permutation(n)

    for u in user_order:
        available = np.ones(k, dtype=bool)
        selected = []
        selected_genres = []
        for _ in range(10):
            utility = scores[u].astype(np.float64).copy()
            if selected_genres:
                genre_gain = np.mean(genres[u, :, None] != np.asarray(selected_genres)[None, :], axis=1)
                utility += gamma_diversity * genre_gain
            # Исторические повторы блокируются, пока есть другие кандидаты.
            repeat_ok = repeat_matrix[u] < 0.5
            exposure_ok = counts[items[u]] < budget[items[u]]
            feasible = available & repeat_ok & exposure_ok
            if not feasible.any(): feasible = available & exposure_ok
            if not feasible.any(): feasible = available
            utility[~feasible] = -np.inf
            chosen = int(np.argmax(utility))
            selected.append(chosen)
            selected_genres.append(int(genres[u, chosen]))
            available[chosen] = False
            counts[int(items[u, chosen])] += 1
        rest = np.flatnonzero(available)
        rest = rest[np.argsort(-scores[u, rest], kind='stable')]
        output[u] = np.asarray(selected + rest.tolist(), dtype=np.int16)
    return output, counts, budget

stage2_bases = pick_stage1_bases(stage1)
stage2_rows = []
stage2_orders = {}
for _, base in tqdm(stage2_bases.iterrows(), total=len(stage2_bases), desc='Pareto stage 2 bases'):
    cfg = {k: float(base[k]) for k in ['alpha_watch', 'beta_choice', 'lambda_popularity', 'mu_ood']}
    scores = static_score(cfg)
    for gamma in [0.0, 0.05, 0.10, 0.20]:
        for budget_relax in [0.0, 0.05, 0.15]:
            for order_seed in [PARETO_SEED, PARETO_SEED + 1]:
                config_id = (f"{base.config_id}_g{gamma:.2f}_b{budget_relax:.2f}_s{order_seed}")
                order, counts, budget = hard_top10_order(
                    scores, items_valid, genres_valid, repeat_valid, gamma, budget_relax, order_seed)
                metrics = policy_metrics(official_cache, order, config_id)
                violations, violation_sum = normalized_guard_violations(metrics)
                overflow_items = int(np.sum(counts > budget))
                row = {
                    'config_id': config_id, **cfg, 'gamma_diversity': gamma,
                    'budget_relax': budget_relax, 'user_order_seed': order_seed,
                    **metrics, 'guard_violation': violation_sum,
                    'hard_feasible': violation_sum == 0.0 and overflow_items == 0,
                    'budget_overflow_items': overflow_items,
                    **{f'v_{k}': v for k, v in violations.items()},
                }
                stage2_rows.append(row)
                stage2_orders[config_id] = order

stage2 = pd.DataFrame(stage2_rows)
stage2.to_csv(PARETO_DIR / 'pareto_stage2_hard_top10.csv', index=False)
display(stage2.sort_values(['guard_violation', 'component_watched_pct'], ascending=[True, False]).head(20))


## 20. Pareto frontier и фиксация validation-конфигурации

Среди feasible-конфигураций выбирается максимальный predicted watch. Если ни одна не проходит ограничения, выбирается не «почти лучший» вариант, а безопасный fallback GPT‑2.


In [ ]:
def pareto_front(frame):
    cols_max = ['component_watched_pct', 'NDCG@10', 'Coverage@10', 'component_diversity']
    cols_min = ['ExposureGini@10', 'component_ood']
    values = frame[cols_max + cols_min].to_numpy(float)
    maximize = np.asarray([1] * len(cols_max) + [-1] * len(cols_min))
    values = values * maximize
    keep = np.ones(len(frame), dtype=bool)
    for i in range(len(frame)):
        if not keep[i]: continue
        dominated = np.all(values >= values[i], axis=1) & np.any(values > values[i], axis=1)
        dominated[i] = False
        if dominated.any(): keep[i] = False
    return frame.loc[keep].copy()

frontier = pareto_front(stage2)
frontier.to_csv(PARETO_DIR / 'pareto_frontier.csv', index=False)
feasible = stage2[stage2.hard_feasible]

if len(feasible):
    selected = feasible.sort_values('component_watched_pct', ascending=False).iloc[0].to_dict()
    selected_order = stage2_orders[selected['config_id']]
    decision = 'best predicted watch among hard-feasible Pareto search policies'
else:
    selected = {
        'config_id': '01_GPT2', 'alpha_watch': 0.0, 'beta_choice': 0.0,
        'lambda_popularity': 0.0, 'mu_ood': 0.0, 'gamma_diversity': 0.0,
        'budget_relax': 0.0, 'user_order_seed': PARETO_SEED,
        **baseline_metrics, 'guard_violation': 0.0, 'hard_feasible': True,
    }
    selected_order = gpt_order
    decision = 'safe fallback to unchanged GPT-2 ordering'

selected_metrics, selected_per_user = policy_metrics(
    official_cache, selected_order, selected['config_id'], return_per_user=True)
dump_json(PARETO_DIR / 'selected_validation_config.json', {
    'decision': decision,
    'config': {k: selected[k] for k in [
        'config_id', 'alpha_watch', 'beta_choice', 'lambda_popularity', 'mu_ood',
        'gamma_diversity', 'budget_relax', 'user_order_seed']},
    'validation_metrics': selected_metrics,
    'guardrails_relative_to': '01_GPT2',
})

final_comparison = pd.DataFrame([baseline_metrics, supervised_metrics, selected_metrics])
display(final_comparison)
final_comparison.to_csv(PARETO_DIR / 'validation_final_comparison.csv', index=False)
print('VALIDATION CONFIG ЗАФИКСИРОВАНА:', selected['config_id'])
print(decision)


## 21. Bootstrap CI выбранной политики

Интервалы считаются парно относительно GPT‑2 на тех же пользователях.


In [ ]:
rng = np.random.default_rng(PARETO_SEED + 100)
ci_rows = []
for metric in ['Recall@10', 'NDCG@10', 'component_watched_pct', 'component_log_choice',
               'component_novelty', 'component_repeat', 'component_ood', 'component_diversity']:
    delta = selected_per_user[metric] - baseline_per_user[metric]
    estimates = np.empty(BOOTSTRAP_REPEATS, dtype=np.float32)
    for b in range(BOOTSTRAP_REPEATS):
        idx = rng.integers(0, len(delta), size=len(delta))
        estimates[b] = delta[idx].mean()
    ci_rows.append({'metric': metric, 'delta_mean': float(delta.mean()),
                    'ci_low': float(np.quantile(estimates, .025)),
                    'ci_high': float(np.quantile(estimates, .975))})
selected_ci = pd.DataFrame(ci_rows)
display(selected_ci)
selected_ci.to_csv(PARETO_DIR / 'selected_bootstrap_ci.csv', index=False)


## 22. Финальный test — намеренно закрыт

Не меняйте коэффициенты после открытия test. Когда validation-конфигурация принята, установите `OPEN_TEST=True` и выполните эту ячейку один раз. Она применит уже сохранённые коэффициенты без нового поиска.


In [ ]:
if OPEN_TEST:
    fixed = json.loads((PARETO_DIR / 'selected_validation_config.json').read_text(encoding='utf-8'))
    cfg = fixed['config']
    if cfg['config_id'] == '01_GPT2':
        print('Validation выбрала GPT-2 fallback; на test сравнивается неизменённая GPT-2.')

    combined_history = pd.concat([interactions['train'], interactions['validation']], ignore_index=True)
    official_test = contexts_from_npz(
        DATA_DIR / 'test_examples.npz', combined_history, EVAL_USERS, SEED + 100,
        boundaries['test']['start'])
    test_top, test_logits, test_target_logits = reference_top_candidates(official_test, CANDIDATE_POOL)
    test_cache = build_candidate_cache(
        official_test, test_top, test_logits, test_target_logits,
        PARETO_DIR / 'candidate_official_test.npz', inject_target=False)

    test_items = test_cache['items'].astype(np.int32)
    test_genres = test_cache['genres'].astype(np.int16)
    test_components = test_cache['components'].astype(np.float32)
    test_sup_scores = model_scores(supervised_model, test_cache)
    test_watch = test_components[:, :, component_index['watched_pct']]
    test_choice = test_components[:, :, component_index['log_choice']]
    test_repeat = test_components[:, :, component_index['repeat']]
    test_ood = test_components[:, :, component_index['ood']]

    # Exposure pressure and budgets строятся только по GPT-2 test recommendations, без target labels.
    test_gpt_order = np.tile(np.arange(CANDIDATE_POOL), (len(test_items), 1))
    test_gpt_counts = np.bincount(test_items[:, :10].ravel(), minlength=N_ITEMS).astype(np.int64)
    pressure = np.log1p(test_gpt_counts).astype(np.float32)
    pressure /= max(float(pressure.max()), 1.0)
    old_counts, old_pressure = gpt_exposure_counts.copy(), gpt_exposure_pressure.copy()
    gpt_exposure_counts[:] = test_gpt_counts
    test_score = (slate_z(test_sup_scores)
                  + cfg['alpha_watch'] * slate_z(test_watch)
                  + cfg['beta_choice'] * slate_z(test_choice)
                  - cfg['lambda_popularity'] * pressure[test_items]
                  - cfg['mu_ood'] * slate_z(test_ood))
    if cfg['config_id'] == '01_GPT2':
        test_selected_order = test_gpt_order
    else:
        test_selected_order, _, _ = hard_top10_order(
            test_score, test_items, test_genres, test_repeat,
            cfg['gamma_diversity'], cfg['budget_relax'], int(cfg['user_order_seed']))
    gpt_exposure_counts[:] = old_counts

    test_rows = [
        policy_metrics(test_cache, test_gpt_order, 'test_GPT2'),
        policy_metrics(test_cache, model_order(supervised_model, test_cache), 'test_supervised'),
        policy_metrics(test_cache, test_selected_order, f"test_{cfg['config_id']}"),
    ]
    test_comparison = pd.DataFrame(test_rows)
    display(test_comparison)
    test_comparison.to_csv(PARETO_DIR / 'test_once_comparison.csv', index=False)
    dump_json(PARETO_DIR / 'TEST_OPENED_ONCE.json', {
        'opened_at_utc': datetime.now(timezone.utc).isoformat(),
        'fixed_validation_config': cfg,
    })
else:
    print('Test закрыт. Сначала оцените Pareto frontier и зафиксируйте validation-конфигурацию.')


## 23. Манифест и архив Pareto-эксперимента


In [ ]:
pareto_manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_run': str(RUN_DIR), 'output_run': str(PARETO_DIR),
    'candidate_model': str(CHECKPOINT_PATH),
    'method': 'supervised reranker + Pareto coefficient sweep + hard Top-10 + global exposure budget',
    'guardrails_relative_to': 'GPT-2',
    'random_configs': PARETO_RANDOM_CONFIGS,
    'test_opened': bool(OPEN_TEST),
    'limitations': [
        'Validation optimizes model-based choice/completion proxies, not causal outcomes.',
        'No impression propensities are available.',
        'Global exposure allocation depends on a fixed user-order seed; two seeds are compared on validation.',
    ],
}
dump_json(PARETO_DIR / 'manifest.json', pareto_manifest)

archive_path = PARETO_DIR.parent / f'{PARETO_NAME}_artifacts.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in PARETO_DIR.rglob('*'):
        if path.is_file() and path.name != 'candidate_official_test.npz':
            zf.write(path, path.relative_to(PARETO_DIR.parent))
print('Pareto-результаты:', PARETO_DIR)
print('Архив:', archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    pass
